# Failure Mode 3: Excessive Steps

> Before starting, read the [project README](../../README.md) for details on failure modes, scorers, and expectations.

The agent completes the task but takes more steps than necessary. The final answer may be correct, but the path to get there is wasteful.

We evaluate this using one approach:

| Scorer | Source | Needs expectations? | What it checks |
|---|---|---|---|
| `ToolCallEfficiency` | MLflow native | No | Are any tool calls unnecessary, or could they have been more efficient? |

For a detailed explanation of this failure mode and how the scorer works internally, see [excessive_steps.md](excessive_steps.md).

### Prerequisites and setup

Start a local MLflow server before running this notebook:

```bash
mlflow server --host 127.0.0.1 --port 5000
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import mlflow
from mlflow.entities import SpanType
from tools import TRAVEL_AGENT_TOOLS, WEATHER_AGENT_TOOLS
from utils import print_eval_results

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("agentic-evaluation")
mlflow.tracing.disable_notebook_display()

EXPERIMENT = mlflow.get_experiment_by_name("agentic-evaluation")

# Clean up old traces for this failure mode
client = mlflow.MlflowClient()
old_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'excessive_steps'",
    return_type="list",
)
if old_traces:
    client.delete_traces(
        experiment_id=EXPERIMENT.experiment_id,
        trace_ids=[t.info.trace_id for t in old_traces],
    )
    print(f"Cleaned up {len(old_traces)} old traces.")

### Create traces

We create synthetic traces demonstrating two sub-cases of excessive steps:

**Redundant calls:** A weather agent answering "What's the weather like in Paris?" calls `get_weather("Paris")` twice (duplicate) and then `web_search` (unnecessary) — 3 calls where 1 would suffice.

**Longer path:** A travel agent booking a flight uses `search_flights` → `get_flight_details` → `book_flight` (3 steps) when `search_and_book` would do it in 1 step. Each call is individually valid — the agent just took a roundabout path.

**Efficient (pass):** Each agent completes its task in a single tool call.

In [ ]:
# --- Failing trace: redundant calls (duplicate + unnecessary) ---
@mlflow.trace(name="weather_agent", span_type=SpanType.AGENT)
def excessive_steps_redundant(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, WEATHER_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "excessive_steps", "expected_result": "fail"}
    )

    with mlflow.start_span(name="get_weather", span_type=SpanType.TOOL) as span:
        span.set_inputs({"location": "Paris"})
        span.set_outputs({
            "temperature_celsius": 22,
            "condition": "partly cloudy",
            "humidity": 65,
        })

    with mlflow.start_span(name="get_weather", span_type=SpanType.TOOL) as span:
        span.set_inputs({"location": "Paris"})
        span.set_outputs({
            "temperature_celsius": 22,
            "condition": "partly cloudy",
            "humidity": 65,
        })

    with mlflow.start_span(name="web_search", span_type=SpanType.TOOL) as span:
        span.set_inputs({"query": "current weather in Paris"})
        span.set_outputs({
            "results": [
                {
                    "title": "Paris Weather",
                    "snippet": "Currently 22°C and partly cloudy.",
                }
            ]
        })

    return "It's currently 22°C and partly cloudy in Paris, with 65% humidity."


# --- Passing trace: efficient weather lookup ---
@mlflow.trace(name="weather_agent", span_type=SpanType.AGENT)
def excessive_steps_efficient_weather(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, WEATHER_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "excessive_steps", "expected_result": "pass"}
    )

    with mlflow.start_span(name="get_weather", span_type=SpanType.TOOL) as span:
        span.set_inputs({"location": "Paris"})
        span.set_outputs({
            "temperature_celsius": 22,
            "condition": "partly cloudy",
            "humidity": 65,
        })

    return "It's currently 22°C and partly cloudy in Paris, with 65% humidity."


# --- Failing trace: longer path (3 steps when 1 would do) ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def excessive_steps_longer_path(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "excessive_steps", "expected_result": "fail"}
    )

    with mlflow.start_span(name="search_flights", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-07-20"})
        span.set_outputs({
            "flights": [{"id": "FL-123", "price": 450, "departure": "08:00"}]
        })

    with mlflow.start_span(name="get_flight_details", span_type=SpanType.TOOL) as span:
        span.set_inputs({"flight_id": "FL-123"})
        span.set_outputs({
            "id": "FL-123",
            "airline": "BA",
            "price": 450,
            "departure": "08:00",
            "arrival": "20:00",
        })

    with mlflow.start_span(name="book_flight", span_type=SpanType.TOOL) as span:
        span.set_inputs({"flight_id": "FL-123"})
        span.set_outputs({"booking_id": "BK-456", "status": "confirmed"})

    return "Your flight is booked! NYC to London, July 20, FL-123, 08:00-20:00. Booking: BK-456."


# --- Passing trace: efficient flight booking ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def excessive_steps_efficient_travel(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "excessive_steps", "expected_result": "pass"}
    )

    with mlflow.start_span(name="search_and_book", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-07-20"})
        span.set_outputs({
            "booking_id": "BK-456",
            "flight_id": "FL-123",
            "airline": "BA",
            "price": 450,
            "departure": "08:00",
            "arrival": "20:00",
            "status": "confirmed",
        })

    return "Your flight is booked! NYC to London, July 20, FL-123, 08:00-20:00. Booking: BK-456."


excessive_steps_redundant([
    {"role": "user", "content": "What's the weather like in Paris right now?"}
])
excessive_steps_efficient_weather([
    {"role": "user", "content": "What's the weather like in Paris right now?"}
])
excessive_steps_longer_path([
    {"role": "user", "content": "Book me a flight from NYC to London on July 20"}
])
excessive_steps_efficient_travel([
    {"role": "user", "content": "Book me a flight from NYC to London on July 20"}
])
mlflow.flush_trace_async_logging()
print("Created 4 traces (2 fail, 2 pass)")

### Load traces

We fetch the Excessive Steps traces — two failing (redundant calls + longer path) and two passing (efficient weather + efficient travel).

In [ ]:
excessive_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'excessive_steps'",
    return_type="list",
)

print(f"Traces found: {len(excessive_traces)}")
for t in excessive_traces:
    tags = t.info.tags or {}
    print(
        f"  [{tags.get('expected_result', '?')}] Input: {str(t.info.request_preview)[:80]}"
    )
    print(f"    Output: {str(t.info.response_preview)[:80]}")
    print()

### Approach 1: ToolCallEfficiency (MLflow native — LLM judge)

`ToolCallEfficiency` reads the user's request, the available tools, and the sequence of tool calls the agent made. It determines whether any tool calls were unnecessary or could have been made more efficient. No expectations needed.

In [ ]:
from mlflow.genai.scorers import ToolCallEfficiency

with mlflow.start_run(run_name="excessive-steps-efficiency") as run:
    results = mlflow.genai.evaluate(
        data=excessive_traces,
        scorers=[ToolCallEfficiency(model="openai:/gpt-4o")],
    )

print_eval_results(results, "tool_call_efficiency", EXPERIMENT.experiment_id)

### Interpreting the results

- **Redundant calls trace** → `no` — the agent called `get_weather("Paris")` twice with identical arguments (duplicate), and also called `web_search` when `get_weather` had already returned the data (unnecessary).
- **Longer path trace** → `no` — the agent used `search_flights` → `get_flight_details` → `book_flight` (3 steps) when `search_and_book` was available and could do it in 1 step.
- **Efficient traces** → `yes` — each agent completed its task in a single tool call.

Note that `ToolCallEfficiency` explicitly treats retries caused by temporary tool failures (timeouts, transient errors) as efficient — it only flags calls that are genuinely unnecessary or suboptimal.

For full details on how the scorer works, see [excessive_steps.md](excessive_steps.md).